# Import Libraries

In [45]:
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Load Dataset

In [46]:
fake = pd.read_csv("/content/drive/MyDrive/Fake_News_Detection/Fake.csv")
true = pd.read_csv("/content/drive/MyDrive/Fake_News_Detection/True.csv")

# Basic Info

In [47]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [48]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


# Label Create

In [49]:
fake['class']=0
true['class']=1

# Combine Both Datasets

In [50]:
df = pd.concat([fake,true], axis=0)

In [51]:
df.sample(10)

,title,text,subject,date,class
11694,Philippine police chief defends deadly drug wa...,MANILA (Reuters) - The police chief of the Phi...,worldnews,"December 20, 2017",1
2802,Trump BETRAYED: GOP Rep. DEFENDS Refugees Aga...,Republican Congressman Charlie Dent represents...,News,"January 28, 2017",0
12889,Support for Irish PM's party surges on back of...,DUBLIN (Reuters) - Irish Prime Minister Leo Va...,worldnews,"December 7, 2017",1
12280,"Putin says Poland should ""grow up"" and stop bl...",MOSCOW/WARSAW (Reuters) - Russian President Vl...,worldnews,"December 14, 2017",1
20238,Macron may see 'slackers' become protest rally...,PARIS (Reuters) - President Emmanuel Macron s ...,worldnews,"September 12, 2017",1
634,Jane Goodall urges U.S. Senate to halt quest f...,WASHINGTON (Reuters) - British primatologist J...,politicsNews,"November 14, 2017",1
20203,"27 Year Old REFUGEE Says He HATES Sweden, Only...",The Left have thrown their women and children ...,left-news,"Jul 29, 2016",0
6713,Factbox: Trump to meet tech leaders from Googl...,(Reuters) - U.S. President-elect Donald Trump ...,politicsNews,"December 14, 2016",1
16238,BOMBSHELL REPORT Proves Trump Right On Illegal...,"According to a nationwide poll, as many as 2 m...",Government News,"Feb 17, 2017",0
5919,This Is How Trump Made Millions While Running...,Trump s business prowess is his major selling ...,News,"June 11, 2016",0


# Drop Unnecessary Columns

In [52]:
df = df.drop(['title','subject','date'], axis=1)

# Index Reset

In [53]:
df.reset_index(inplace=True)

In [54]:
df.drop(['index'],axis=1,inplace=True)

In [55]:
df.sample(5)

,text,class
10474,"Guess how leftists, millennials and Black Liv...",0
2155,"At a town hall in Frost, Texas last week a vot...",0
44215,CHISINAU (Reuters) - The Moldovan government s...,1
17474,Jenna Fischer is best known for playing the ve...,0
44740,SEOUL (Reuters) - South Korean President Moon ...,1


# Text Cleaning

In [56]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', "", text)
    text = re.sub(r'\W', " ", text)
    text = re.sub(r'https?://\S+|www\.\S+', "", text)
    text = re.sub(r'<.*?>+', "", text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), "", text)
    text = re.sub(r'\n', "", text)
    text = re.sub(r'\w*\d\w*', "", text)
    return text

In [57]:
df['text'] = df['text'].apply(clean_text)


# Features and Target

In [58]:
X=df['text']
y=df['class']

# Train Test Split

In [59]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)

# Convert Text into Numbers with TF-IDF

In [60]:
vectorizer = TfidfVectorizer()
xv_train = vectorizer.fit_transform(X_train)
xv_test = vectorizer.transform(X_test)

# Model Training

In [61]:
model = LogisticRegression()
model.fit(xv_train,y_train)

LogisticRegression()

# Prediction

In [62]:
prediction = model.predict(xv_test)

# Accuracy Check

In [63]:
model.score(xv_test,y_test)

0.9858351893095768

# Detailed Report

In [64]:
print(classification_report(y_test,prediction))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5895
           1       0.98      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



# Model Save

In [65]:
import joblib

joblib.dump(vectorizer,"/content/drive/MyDrive/Fake_News_Detection/vectorizer.jb")
joblib.dump(model,"/content/drive/MyDrive/Fake_News_Detection/model.jb")

['/content/drive/MyDrive/Fake_News_Detection/model.jb']